# Olive Cartographer consolidated analysis for manuscript

This notebook replaces the previous two notebooks and resolves the **120 vs 60** issue.

**Data logic**

- The raw CSV contains **120 side-level observations**: 20 rows × 3 longitudinal sections × 2 canopy faces.
- The manuscript should use **60 subsection-level observations**: 20 rows × 3 sections, after combining the two canopy faces.
- This is the correct unit for the paper because production is available/validated after both canopy faces are combined.

The notebook exports the figures used by the LaTeX manuscript and prints manuscript-ready statistics using the final Cartographer terminology.


In [1]:
# Core packages
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / 'Atlas_data_new.csv'
FIG_DIR = BASE_DIR / 'figures'
TAB_DIR = BASE_DIR / 'tables'
FIG_DIR.mkdir(exist_ok=True)
TAB_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams['figure.dpi'] = 150
# Use same palette as analysis.ipynb for consistent figures
PALETTE = sns.color_palette('deep')

# Manuscript display labels. Internal column names are kept unchanged so the data pipeline remains stable.
DISPLAY_LABELS = {
    'MI_Garcia': 'Cartographer MI',
    'MI_Visual': 'Manual MI',
    'Color Index': 'Cartographer CI',
    'Atlas_estimated_production_kg': 'Cartographer production measure',
    'Measured_production_kg': 'Field-measured production',
}

def label_for(name):
    return DISPLAY_LABELS.get(name, name)


## 1. Load raw side-level data

Place `Elaborate_Roberto_light.csv` in the same folder as this notebook.

Expected raw unit: **one canopy face of one row-section**. Therefore the raw file should have 120 rows.


In [2]:
if not DATA_FILE.exists():
    raise FileNotFoundError(f'Missing {DATA_FILE.name}. Put it in the same folder as this notebook.')

# Read the raw data from the Excel file.
raw = pd.read_excel(DATA_FILE.with_suffix('.xlsx'), sheet_name='Atlas_data', header=0)
raw.columns = [c.strip() for c in raw.columns]

# Standardise maturity-index column names from the existing files.
rename_map = {
    'M.I. NORM': 'MI_Garcia',
    # 'M.I. Garcia': 'MI_Garcia',
    'M.I. visivo': 'MI_Visual',
    'Produzione plot [kg]': 'Production_plot_kg',
    'Produzione plot somma dei due lati [kg]': 'Atlas_production_both_faces_kg',
    'Produzione plot MISURATA': 'Measured_production_kg',    
}
raw = raw.rename(columns={k:v for k,v in rename_map.items() if k in raw.columns})

print('Raw side-level dataset shape:', raw.shape)
print('Columns:')
print(list(raw.columns))
raw.head()


Raw side-level dataset shape: (120, 20)
Columns:
['Plot', 'Riga', 'frutti/pianta', 'N. frutti', 'Superficie plot [ha]', 'Production_plot_kg', 'Resa [kg/ha]', 'Atlas_production_both_faces_kg', 'Resa somma dei due lati [kg/ha]', 'Measured_production_kg', 'Color Index', 'CL 0', 'CL 1', 'CL 2', 'CL 3', 'CL 4', 'M.I. Garcia', 'MI_Garcia', 'MI_Visual', 'Color Index.1']


,Plot,Riga,frutti/pianta,N. frutti,Superficie plot [ha],Production_plot_kg,Resa [kg/ha],Atlas_production_both_faces_kg,Resa somma dei due lati [kg/ha],Measured_production_kg,Color Index,CL 0,CL 1,CL 2,CL 3,CL 4,M.I. Garcia,MI_Garcia,MI_Visual,Color Index.1
0,1,1,1910,74353,0.02,211.820,6617.4,211.820,13487.238,715.74476,0.62,47,73,30,24,0,2.05,1.18,1.91,0.62
1,1,2,1982,77189,0.02,198.120,6869.8,198.120,NaN,NaN,0.67,46,49,26,42,0,2.27,1.39,2.03,0.67
2,1,3,1320,51392,0.02,149.900,4573.9,149.900,8868.761,541.37522,0.78,17,38,29,84,0,3.48,2.07,2.31,0.78
3,1,4,1246,48257,0.02,174.175,4294.9,174.175,NaN,NaN,0.85,12,20,34,101,0,3.91,2.34,2.31,0.85
4,1,5,1324,51553,0.02,180.400,4588.2,180.400,,645.25172,0.78,28,28,22,85,0,3.27,2.01,2.21,0.78


## 3. Descriptive statistics and ripening-class distribution

In [3]:
raw[['N. frutti', 'Production_plot_kg', 'Color Index', 'MI_Garcia', 'MI_Visual']].describe().round(2)

,N. frutti,Production_plot_kg,Color Index,MI_Garcia,MI_Visual
count,120.00,120.00,120.00,120.00,120.00
mean,75710.51,207.18,0.84,2.42,2.38
std,26184.95,52.08,0.07,0.41,0.18
min,30893.00,122.40,0.61,1.18,1.73
25%,54762.00,174.03,0.81,2.32,2.31
50%,72444.00,196.03,0.85,2.54,2.42
75%,86926.25,223.58,0.89,2.71,2.49
max,160409.00,400.64,0.93,2.94,2.69


In [4]:
CL_COLS = ['CL 0','CL 1','CL 2','CL 3','CL 4']

class_totals = raw[CL_COLS].sum()
detected_fruit = raw['N. frutti'].sum() if 'N. frutti' in raw.columns else None

class_pct = class_totals / class_totals.sum() * 100
class_detected_fruit = class_pct * detected_fruit / 100 if detected_fruit is not None else None
class_summary = pd.DataFrame({'class': CL_COLS, 'percent': class_pct.values, 'detected_fruit' : class_detected_fruit.values.astype(int)})
class_summary.to_csv(TAB_DIR / 'ripening_class_distribution.csv', index=False)

print('Ripening class distribution after aggregating both faces (n=60 subsections):')
print(class_summary.to_string(index=False, formatters={'percent': '{:.1f}'.format}))
print('Total Cartographer-detected olives:', int(detected_fruit))
    
# Use Seaborn barplot for consistent styling
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x='class', y='detected_fruit', data=class_summary, palette=[PALETTE[0]]*len(class_summary), ax=ax, edgecolor='black', linewidth=1.5)
for i, row in class_summary.iterrows():
    ax.text(i, row['detected_fruit'], f"{row['percent']:.1f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xlabel('MI class', fontsize=12, fontweight='bold')
ax.set_ylabel('Detected olive count', fontsize=12, fontweight='bold')
ax.set_title('Cartographer-detected olives per ripening class (n=60 subsections)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
fig.savefig(FIG_DIR / 'corr_fig4_olive_count_per_class.pdf', dpi=300, bbox_inches='tight')
plt.show()


Ripening class distribution after aggregating both faces (n=60 subsections):
class percent  detected_fruit
 CL 0     5.4          492982
 CL 1    10.9          990521
 CL 2    21.5         1949661
 CL 3    62.2         5652095
 CL 4     0.0               0
Total Cartographer-detected olives: 9085261


## 4. Correlation matrix on the 120 subsection-level dataset


In [5]:
raw.columns

Index(['Plot', 'Riga', 'frutti/pianta', 'N. frutti', 'Superficie plot [ha]',
       'Production_plot_kg', 'Resa [kg/ha]', 'Atlas_production_both_faces_kg',
       'Resa somma dei due lati [kg/ha]', 'Measured_production_kg',
       'Color Index', 'CL 0', 'CL 1', 'CL 2', 'CL 3', 'CL 4', 'M.I. Garcia',
       'MI_Garcia', 'MI_Visual', 'Color Index.1'],
      dtype='object')

In [6]:
corr_vars = [c for c in ['MI_Garcia','MI_Visual','Color_Index'] if c in raw.columns]
if len(corr_vars) < 2:
    raise ValueError('Not enough variables available for correlation analysis.')

data_corr = raw[corr_vars].dropna()
print('Correlation dataset n =', len(data_corr))

r_mat = pd.DataFrame(np.eye(len(corr_vars)), index=corr_vars, columns=corr_vars)
p_mat = pd.DataFrame(np.zeros((len(corr_vars), len(corr_vars))), index=corr_vars, columns=corr_vars)

for i,a in enumerate(corr_vars):
    for j,b in enumerate(corr_vars):
        if i < j:
            r,p = stats.pearsonr(data_corr[a], data_corr[b])
            r_mat.loc[a,b] = r_mat.loc[b,a] = r
            p_mat.loc[a,b] = p_mat.loc[b,a] = p
            print(f'{label_for(a)} vs {label_for(b)}: r={r:.4f}, p={p:.3e}')

# Plot annotated matrix.
def star(p):
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return 'ns'

annot = r_mat.copy().astype(str)
for a in corr_vars:
    for b in corr_vars:
        if a == b:
            annot.loc[a,b] = '1.00'
        else:
            annot.loc[a,b] = f"{r_mat.loc[a,b]:.2f}\n{star(p_mat.loc[a,b])}"

plot_r_mat = r_mat.rename(index=DISPLAY_LABELS, columns=DISPLAY_LABELS)
plot_annot = annot.rename(index=DISPLAY_LABELS, columns=DISPLAY_LABELS)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(
    plot_r_mat.astype(float),
    annot=plot_annot,
    fmt='',
    center=0.95,
    vmin=0.9,
    vmax=1.0,
    square=True,
    cbar_kws={'label':'Pearson r', 'ticks':[0.90, 0.92, 0.94, 0.96, 0.98, 1.00]},
    cmap='viridis',
    ax=ax,
    annot_kws={"size": 15, "weight": "bold"},
    linewidths=0.5,
    linecolor='gray'
)

ax.set_title(
    "Pearson correlation among Cartographer maturity indicators\n(* p<0.05  ** p<0.01  *** p<0.001)", 
    fontsize=13,
    fontweight="bold",
    pad=15
)

# Increase x/y axis labels
ax.set_xticklabels(ax.get_xticklabels(), fontsize=11, rotation=45, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), fontsize=11, rotation=0)

# Increase colorbar font
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=13)
cbar.set_label('Pearson r', fontsize=14, fontweight='bold')

fig.tight_layout()
fig.savefig(FIG_DIR / 'corr_fig1_correlation_matrix.pdf', dpi=300, bbox_inches='tight')
plt.show()


Correlation dataset n = 120
Cartographer MI vs Manual MI: r=0.9006, p=1.626e-44


## 6. Helper functions for regression, cross-validation and Bland--Altman


In [7]:
def assign_spatial_folds(df, row_col='Riga', plot_col='Plot', k=5):
    """Assign adjacent grid columns to contiguous spatial CV folds."""
    missing = [c for c in [row_col, plot_col] if c not in df.columns]
    if missing:
        raise KeyError(f'Missing grid-coordinate columns: {missing}')

    out = df.copy()
    ordered_rows = np.sort(out[row_col].dropna().unique())
    if len(ordered_rows) < k:
        raise ValueError(f'{row_col} has {len(ordered_rows)} unique values; at least {k} are required.')

    # np.array_split keeps neighbouring Riga values together and balances fold sizes.
    row_bands = np.array_split(ordered_rows, k)
    fold_lookup = {row: fold for fold, band in enumerate(row_bands, start=1) for row in band}
    out['spatial_fold'] = out[row_col].map(fold_lookup).astype('Int64')
    return out


def regression_with_cv(df, x_col, y_col, k=5, row_col='Riga', plot_col='Plot'):
    """Fit a full model and evaluate it with contiguous five-fold spatial CV."""
    required = [x_col, y_col, row_col, plot_col]
    d = df[required].dropna().copy()
    d = assign_spatial_folds(d, row_col=row_col, plot_col=plot_col, k=k)

    X = d[[x_col]].to_numpy()
    y = d[y_col].to_numpy()
    model = LinearRegression().fit(X, y)
    pred = model.predict(X)
    r2 = r2_score(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    r, p = stats.pearsonr(d[x_col], d[y_col])

    oof_pred = np.full(len(d), np.nan)
    fold_metrics = []
    for fold in range(1, k + 1):
        test_mask = d['spatial_fold'].eq(fold).to_numpy()
        train_mask = ~test_mask
        fold_model = LinearRegression().fit(X[train_mask], y[train_mask])
        fold_pred = fold_model.predict(X[test_mask])
        oof_pred[test_mask] = fold_pred
        fold_metrics.append({
            'fold': fold,
            'n_train': int(train_mask.sum()),
            'n_test': int(test_mask.sum()),
            'r2': r2_score(y[test_mask], fold_pred),
            'rmse': np.sqrt(mean_squared_error(y[test_mask], fold_pred)),
        })

    fold_metrics = pd.DataFrame(fold_metrics)
    d['cv_prediction'] = oof_pred
    return {
        'data': d, 'model': model, 'pred': pred,
        'slope': float(model.coef_[0]), 'intercept': float(model.intercept_),
        'r': float(r), 'p': float(p), 'r2': float(r2), 'rmse': float(rmse),
        'cv_r2_mean': float(fold_metrics['r2'].mean()),
        'cv_r2_sd': float(fold_metrics['r2'].std(ddof=1)),
        'cv_rmse_mean': float(fold_metrics['rmse'].mean()),
        'cv_rmse_sd': float(fold_metrics['rmse'].std(ddof=1)),
        'cv_oof_r2': float(r2_score(y, oof_pred)),
        'cv_oof_rmse': float(np.sqrt(mean_squared_error(y, oof_pred))),
        'fold_metrics': fold_metrics,
    }


def _draw_fold_boundaries(ax, folded_df, row_col='Riga'):
    columns = np.sort(folded_df[row_col].dropna().unique())
    fold_by_column = folded_df.groupby(row_col)['spatial_fold'].first().reindex(columns).to_numpy()
    changes = np.flatnonzero(fold_by_column[1:] != fold_by_column[:-1]) + 1
    for boundary in changes:
        ax.axvline(boundary, color='white', linewidth=4)
        ax.axvline(boundary, color='black', linewidth=1.2)


def plot_spatial_cv_folds(df, row_col='Riga', plot_col='Plot', k=5):
    folded = assign_spatial_folds(df, row_col=row_col, plot_col=plot_col, k=k)
    grid = folded.pivot(index=plot_col, columns=row_col, values='spatial_fold').sort_index(ascending=False).astype(float)

    fig, ax = plt.subplots(figsize=(15, 5))
    sns.heatmap(
        grid, cmap=sns.color_palette('viridis', as_cmap=True),
        vmin=0.5, vmax=k + 0.5, annot=True, fmt='.0f',
        linewidths=0.5, linecolor='white', square=True,
        cbar_kws={'label': 'Spatial CV fold', 'ticks': range(1, k + 1)}, ax=ax
    )
    _draw_fold_boundaries(ax, folded, row_col=row_col)
    ax.set_title('Five contiguous spatial cross-validation folds', fontsize=14, fontweight='bold')
    ax.set_xlabel('Riga (grid column)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Plot (grid row)', fontsize=12, fontweight='bold')
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'spatial_cv_5fold_layout.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    return folded


def plot_spatial_values_by_fold(folded, row_col='Riga', plot_col='Plot'):
    value_specs = [
        ('N. frutti', 'Detected fruit count'),
        ('Production_plot_kg', 'Fruit weight / plot production (kg)'),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(18, 4.2), constrained_layout=True)

    for ax, (value_col, title) in zip(axes, value_specs):
        if value_col not in folded.columns:
            raise KeyError(f'Missing value column: {value_col}')
        grid = folded.pivot(index=plot_col, columns=row_col, values=value_col).sort_index(ascending=False)
        sns.heatmap(
            grid, cmap='viridis', linewidths=0.35, linecolor='white', square=True,
            cbar_kws={'label': title}, ax=ax
        )
        _draw_fold_boundaries(ax, folded, row_col=row_col)
        ax.set_title(f'{title} across the five spatial folds', fontsize=13, fontweight='bold')
        ax.set_xlabel('Riga (grid column)', fontsize=11, fontweight='bold')
        ax.set_ylabel('Plot (grid row)', fontsize=11, fontweight='bold')

    fig.savefig(FIG_DIR / 'spatial_cv_fruit_count_and_weight.pdf', dpi=300, bbox_inches='tight')
    plt.show()


CV_METHOD_LABELS = {
    'random_5fold': 'Random 5-fold',
    'spatial_block_5fold': 'Spatial block 5-fold',
    'leave_one_row_out': 'Leave-one-Riga-out',
}


def _evaluate_cv_splits(X, y, splits, fold_labels):
    """Return cumulative out-of-fold predictions and fold-level metrics."""
    oof_prediction = np.full(len(y), np.nan, dtype=float)
    fold_assignment = np.full(len(y), -1, dtype=int)
    rows = []

    for fold_label, (train_index, test_index) in zip(fold_labels, splits):
        model = LinearRegression().fit(X[train_index], y[train_index])
        prediction = model.predict(X[test_index])
        oof_prediction[test_index] = prediction
        fold_assignment[test_index] = int(fold_label)
        rows.append({
            'fold': int(fold_label),
            'n_train': len(train_index),
            'n_test': len(test_index),
            'r2': r2_score(y[test_index], prediction),
            'rmse': np.sqrt(mean_squared_error(y[test_index], prediction)),
        })

    if np.isnan(oof_prediction).any():
        raise RuntimeError('Some observations did not receive an out-of-fold prediction.')

    return oof_prediction, fold_assignment, pd.DataFrame(rows)


def regression_with_validation(df, x_col, y_col, k=5, row_col='Riga', plot_col='Plot', random_state=42):
    """Compare random CV, contiguous spatial blocks, and leave-one-row-out CV."""
    required = [x_col, y_col, row_col, plot_col]
    d = df[required].dropna().copy().reset_index().rename(columns={'index': 'source_index'})
    d = assign_spatial_folds(d, row_col=row_col, plot_col=plot_col, k=k)

    X = d[[x_col]].to_numpy(dtype=float)
    y = d[y_col].to_numpy(dtype=float)
    full_model = LinearRegression().fit(X, y)
    full_prediction = full_model.predict(X)
    r, p = stats.pearsonr(d[x_col], d[y_col])

    random_cv = KFold(n_splits=k, shuffle=True, random_state=random_state)
    random_splits = list(random_cv.split(X, y))

    block_splits = []
    for fold in range(1, k + 1):
        test_index = np.flatnonzero(d['spatial_fold'].eq(fold).to_numpy())
        train_index = np.flatnonzero(~d['spatial_fold'].eq(fold).to_numpy())
        block_splits.append((train_index, test_index))

    row_groups = d[row_col].to_numpy()
    logo = LeaveOneGroupOut()
    logo_splits = list(logo.split(X, y, groups=row_groups))
    logo_labels = [row_groups[test_index][0] for _, test_index in logo_splits]

    split_definitions = {
        'random_5fold': (random_splits, range(1, k + 1)),
        'spatial_block_5fold': (block_splits, range(1, k + 1)),
        'leave_one_row_out': (logo_splits, logo_labels),
    }

    fold_metrics = {}
    summary_rows = []
    for method, (splits, labels) in split_definitions.items():
        oof, assignment, metrics = _evaluate_cv_splits(X, y, splits, labels)
        d[f'cv_fold_{method}'] = assignment
        d[f'pred_{method}'] = oof
        d[f'resid_{method}'] = y - oof
        metrics.insert(0, 'method', method)
        fold_metrics[method] = metrics
        summary_rows.append({
            'method': method,
            'method_label': CV_METHOD_LABELS[method],
            'n_splits': len(metrics),
            'oof_r2': r2_score(y, oof),
            'oof_rmse': np.sqrt(mean_squared_error(y, oof)),
            'fold_r2_mean': metrics['r2'].mean(),
            'fold_r2_sd': metrics['r2'].std(ddof=1),
            'fold_rmse_mean': metrics['rmse'].mean(),
            'fold_rmse_sd': metrics['rmse'].std(ddof=1),
        })

    validation_summary = pd.DataFrame(summary_rows)
    spatial = validation_summary.set_index('method').loc['spatial_block_5fold']
    return {
        'data': d, 'model': full_model, 'pred': full_prediction,
        'slope': float(full_model.coef_[0]), 'intercept': float(full_model.intercept_),
        'r': float(r), 'p': float(p),
        'r2': float(r2_score(y, full_prediction)),
        'rmse': float(np.sqrt(mean_squared_error(y, full_prediction))),
        'validation_summary': validation_summary,
        'fold_metrics': fold_metrics,
        # Backward-compatible aliases refer to the five spatial blocks.
        'cv_r2_mean': float(spatial['fold_r2_mean']),
        'cv_r2_sd': float(spatial['fold_r2_sd']),
        'cv_rmse_mean': float(spatial['fold_rmse_mean']),
        'cv_rmse_sd': float(spatial['fold_rmse_sd']),
        'cv_oof_r2': float(spatial['oof_r2']),
        'cv_oof_rmse': float(spatial['oof_rmse']),
    }


def moran_i_grid(df, value_col, row_col='Riga', plot_col='Plot', permutations=9999, seed=42):
    """Global Moran's I with rook adjacency and a two-sided permutation test."""
    d = df[[row_col, plot_col, value_col]].dropna().copy()
    values = d[value_col].to_numpy(dtype=float)
    coordinates = d[[row_col, plot_col]].to_numpy(dtype=float)
    delta_row = np.abs(coordinates[:, None, 0] - coordinates[None, :, 0])
    delta_plot = np.abs(coordinates[:, None, 1] - coordinates[None, :, 1])
    weights = (((delta_row == 1) & (delta_plot == 0)) |
               ((delta_row == 0) & (delta_plot == 1)))
    np.fill_diagonal(weights, False)
    focal, neighbour = np.nonzero(weights)
    s0 = len(focal)
    n = len(values)
    expected_i = -1 / (n - 1)

    def statistic(v):
        z = v - v.mean()
        denominator = np.dot(z, z)
        return (n / s0) * np.dot(z[focal], z[neighbour]) / denominator

    observed_i = statistic(values)
    rng = np.random.default_rng(seed)
    simulated = np.array([statistic(rng.permutation(values)) for _ in range(permutations)])
    p_value = (1 + np.sum(np.abs(simulated - expected_i) >=
                          np.abs(observed_i - expected_i))) / (permutations + 1)
    return {
        'moran_i': float(observed_i),
        'expected_i': float(expected_i),
        'p_permutation_two_sided': float(p_value),
        'permutations': int(permutations),
        'adjacency': 'rook',
    }


def residual_moran_table(result, row_col='Riga', plot_col='Plot', permutations=9999):
    rows = []
    for method, label in CV_METHOD_LABELS.items():
        moran = moran_i_grid(
            result['data'], f'resid_{method}', row_col=row_col, plot_col=plot_col,
            permutations=permutations, seed=42
        )
        rows.append({'method': method, 'method_label': label, **moran})
    return pd.DataFrame(rows)


def export_validation_outputs(result, prefix, row_col='Riga', plot_col='Plot'):
    moran = residual_moran_table(result, row_col=row_col, plot_col=plot_col)
    result['validation_summary'].to_csv(TAB_DIR / f'{prefix}_cv_method_comparison.csv', index=False)
    pd.concat(result['fold_metrics'].values(), ignore_index=True).to_csv(
        TAB_DIR / f'{prefix}_cv_fold_metrics_all_methods.csv', index=False
    )
    result['data'].to_csv(TAB_DIR / f'{prefix}_cumulative_oof_predictions.csv', index=False)
    moran.to_csv(TAB_DIR / f'{prefix}_residual_moran_i.csv', index=False)
    return moran


def print_validation_report(result, moran):
    display_columns = [
        'method_label', 'n_splits', 'oof_r2', 'oof_rmse',
        'fold_r2_mean', 'fold_r2_sd', 'fold_rmse_mean', 'fold_rmse_sd'
    ]
    print('Validation comparison using cumulative out-of-fold predictions:')
    print(result['validation_summary'][display_columns].round(4).to_string(index=False))
    print("\nMoran's I of cumulative OOF residuals (rook adjacency):")
    print(moran[['method_label', 'moran_i', 'expected_i', 'p_permutation_two_sided']].round(4).to_string(index=False))


def format_permutation_p(p_value):
    return 'p<0.001' if p_value < 0.001 else f'p={p_value:.3f}'


def plot_cv_comparison(result, moran, y_col, ylabel, filename):
    d = result['data']
    methods = list(CV_METHOD_LABELS)
    colors = sns.color_palette('viridis', n_colors=len(methods))
    fig, axes = plt.subplots(2, 3, figsize=(16, 9.5), constrained_layout=True)

    for ax, method, color in zip(axes[0], methods, colors):
        pred_col = f'pred_{method}'
        sns.scatterplot(data=d, x=y_col, y=pred_col, color=color, s=52, alpha=0.8, ax=ax)
        low = min(d[y_col].min(), d[pred_col].min())
        high = max(d[y_col].max(), d[pred_col].max())
        ax.plot([low, high], [low, high], '--', color='black', linewidth=1.4)
        metric = result['validation_summary'].set_index('method').loc[method]
        ax.set_title(f"{CV_METHOD_LABELS[method]}\nOOF R²={metric['oof_r2']:.3f}, RMSE={metric['oof_rmse']:.3f}", fontweight='bold')
        ax.set_xlabel(f'Observed {ylabel}')
        ax.set_ylabel(f'OOF-predicted {ylabel}')

    summary = result['validation_summary']
    sns.barplot(data=summary, x='method_label', y='oof_r2', hue='method_label', palette='viridis', legend=False, ax=axes[1, 0])
    axes[1, 0].axhline(0, color='black', linewidth=1)
    axes[1, 0].set_title('Cumulative OOF R²', fontweight='bold')
    axes[1, 0].set_xlabel('')
    axes[1, 0].set_ylabel('OOF R²')
    axes[1, 0].tick_params(axis='x', rotation=20)

    sns.barplot(data=summary, x='method_label', y='oof_rmse', hue='method_label', palette='viridis', legend=False, ax=axes[1, 1])
    axes[1, 1].set_title('Cumulative OOF RMSE', fontweight='bold')
    axes[1, 1].set_xlabel('')
    axes[1, 1].set_ylabel('OOF RMSE')
    axes[1, 1].tick_params(axis='x', rotation=20)

    sns.barplot(data=moran, x='method_label', y='moran_i', hue='method_label', palette='viridis', legend=False, ax=axes[1, 2])
    axes[1, 2].axhline(0, color='black', linewidth=1)
    axes[1, 2].set_title("Moran's I of OOF residuals", fontweight='bold')
    axes[1, 2].set_xlabel('')
    axes[1, 2].set_ylabel("Moran's I")
    axes[1, 2].tick_params(axis='x', rotation=20)
    for patch, p_value in zip(axes[1, 2].patches, moran['p_permutation_two_sided']):
        axes[1, 2].text(patch.get_x() + patch.get_width()/2, patch.get_height(), format_permutation_p(p_value), ha='center', va='bottom', fontsize=9)

    fig.suptitle(f'Cross-validation comparison: {ylabel}', fontsize=16, fontweight='bold')
    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()


def plot_observations_and_residuals(result, moran, y_col, ylabel, filename, row_col='Riga', plot_col='Plot'):
    d = result['data']
    methods = list(CV_METHOD_LABELS)
    residual_columns = [f'resid_{method}' for method in methods]
    residual_limit = max(np.abs(d[column]).max() for column in residual_columns)
    moran_lookup = moran.set_index('method')

    fig, axes = plt.subplots(2, 2, figsize=(15, 8), constrained_layout=True)
    observed_grid = d.pivot(index=plot_col, columns=row_col, values=y_col).sort_index(ascending=False)
    sns.heatmap(
        observed_grid, cmap='viridis', square=True, linewidths=0.35, linecolor='white',
        cbar_kws={'label': f'Observed {ylabel}'}, ax=axes[0, 0]
    )
    axes[0, 0].set_title(f'Observed {ylabel}', fontweight='bold')

    for ax, method in zip([axes[0, 1], axes[1, 0], axes[1, 1]], methods):
        residual_grid = d.pivot(index=plot_col, columns=row_col, values=f'resid_{method}').sort_index(ascending=False)
        sns.heatmap(
            residual_grid, cmap='vlag', center=0, vmin=-residual_limit, vmax=residual_limit,
            square=True, linewidths=0.35, linecolor='white',
            cbar_kws={'label': 'Observed - OOF prediction'}, ax=ax
        )
        row = moran_lookup.loc[method]
        ax.set_title(
            f"{CV_METHOD_LABELS[method]} residuals\nMoran's I={row['moran_i']:.3f}, {format_permutation_p(row['p_permutation_two_sided'])}",
            fontweight='bold'
        )

    for ax in axes.flat:
        ax.set_xlabel('Riga')
        ax.set_ylabel('Plot')

    fig.suptitle(f'Spatial distribution of observations and OOF residuals: {ylabel}', fontsize=16, fontweight='bold')
    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()


def plot_regression(
    res, x_col, y_col, filename, xlabel, ylabel,
    identity=False, x_log=False, y_log=False,
    force_origin=True,
    zoom_origin=False
):
    d = res['data'].copy()

    if x_log and (d[x_col] <= 0).any():
        raise ValueError(f"x_log=True requires all values in '{x_col}' to be > 0")

    if y_log and (d[y_col] <= 0).any():
        raise ValueError(f"y_log=True requires all values in '{y_col}' to be > 0")

    fig, ax = plt.subplots(figsize=(6.5, 5.2))

    sns.regplot(
        x=x_col,
        y=y_col,
        data=d,
        scatter_kws={
            'alpha': 0.75,
            'color': PALETTE[0],
            's': 55,
            'edgecolor': 'white'
        },
        line_kws={
            'color': PALETTE[1],
            'linewidth': 2.5
        },
        ci=95,
        ax=ax
    )

    if ax.collections:
        ax.collections[0].set_label('Observed')

    if ax.lines:
        ax.lines[0].set_label('Linear fit')

    if x_log:
        ax.set_xscale('log')

    if y_log:
        ax.set_yscale('log')

    # Force the visual origin only on linear axes
    if force_origin and not x_log:
        ax.set_xlim(left=0)

    if force_origin and not y_log:
        ax.set_ylim(bottom=0)

    # Add identity line only when x and y are in the same unit
    if identity:
        if xlabel != ylabel:
            raise ValueError(
                "Identity line is only meaningful when x and y have the same unit."
            )

        mn = min(d[x_col].min(), d[y_col].min())
        mx = max(d[x_col].max(), d[y_col].max())

        if x_log or y_log:
            mn = max(mn, min(d[x_col][d[x_col] > 0].min(), d[y_col][d[y_col] > 0].min()))

        ax.plot(
            [mn, mx],
            [mn, mx],
            linestyle='--',
            color=PALETTE[2],
            linewidth=1.8,
            label='Identity line'
        )

    ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')

    ax.set_title(
        f"{ylabel} vs {xlabel}",
        fontsize=13,
        fontweight='bold',
        pad=12
    )

    eq_text = f"y = {res['slope']:.4f}x {res['intercept']:+.4f}"
    stats_text = (
        f"{eq_text}\n"
        f"r = {res['r']:.3f}, p = {res['p']:.3e}\n"
        f"R² = {res['r2']:.3f}"
    )

    ax.text(
        0.05,
        0.95,
        stats_text,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85),
        family='monospace'
    )

    ax.grid(True, alpha=0.25)
    ax.legend(loc='lower right', fontsize=10, frameon=True)

    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
    
def bland_altman(a, b):
    a = pd.Series(a).astype(float)
    b = pd.Series(b).astype(float)
    d = pd.DataFrame({'a': a, 'b': b}).dropna()
    mean_vals = (d['a'] + d['b']) / 2
    diff_vals = d['a'] - d['b']
    bias = diff_vals.mean()
    sd = diff_vals.std(ddof=1)
    loa_low = bias - 1.96 * sd
    loa_high = bias + 1.96 * sd
    sw = stats.shapiro(diff_vals) if len(diff_vals) >= 3 else (np.nan, np.nan)
    prop_r, prop_p = stats.pearsonr(mean_vals, diff_vals) if len(diff_vals) >= 3 else (np.nan, np.nan)
    return {
        'mean': mean_vals, 'diff': diff_vals,
        'bias': float(bias), 'sd': float(sd), 'loa_low': float(loa_low), 'loa_high': float(loa_high),
        'shapiro_w': float(sw[0]) if not np.isnan(sw[0]) else np.nan, 'shapiro_p': float(sw[1]) if not np.isnan(sw[1]) else np.nan,
        'prop_r': float(prop_r) if not np.isnan(prop_r) else np.nan, 'prop_p': float(prop_p) if not np.isnan(prop_p) else np.nan,
    }


def plot_bland_altman(res, filename, xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    sns.scatterplot(x=res['mean'], y=res['diff'], alpha=0.7, color=PALETTE[0], s=60, ax=ax)
    ax.axhline(res['bias'], linewidth=2.5, label=f"bias={res['bias']:.3f}", color='black')
    ax.axhline(res['loa_low'], linestyle='--', label=f"LoA low={res['loa_low']:.3f}", color=PALETTE[3], linewidth=2)
    ax.axhline(res['loa_high'], linestyle='--', label=f"LoA high={res['loa_high']:.3f}", color=PALETTE[3], linewidth=2)
    ax.fill_between(ax.get_xlim(), res['loa_low'], res['loa_high'], color=PALETTE[3], alpha=0.1)
    ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(f"Bland–Altman: {ylabel}", fontsize=13, fontweight='bold', pad=12)
    
    stats_text = f"Bias = {res['bias']:.3f}\nLoA = [{res['loa_low']:.3f}, {res['loa_high']:.3f}]\nShapiro p = {res['shapiro_p']:.3e}\nProp r = {res['prop_r']:.3f}, p = {res['prop_p']:.3e}"
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            family='monospace')
    
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()


## 7. Regression, spatial diagnostics, and cross-validation comparison

All 120 subsections come from the same orchard and acquisition date. Random five-fold CV is therefore retained only as an internal benchmark and is compared with contiguous spatial blocks and leave-one-`Riga`-out validation. Cumulative out-of-fold predictions are reported for every observation, and Moran's I is calculated on OOF residuals using rook-adjacent grid cells and 9,999 permutations. These analyses assess internal spatial robustness; they do **not** establish transferability to another orchard, date, cultivar, or acquisition campaign.


In [8]:
# Visualise the same five contiguous folds used by both regressions.
spatial_grid = plot_spatial_cv_folds(raw, row_col='Riga', plot_col='Plot', k=5)
plot_spatial_values_by_fold(spatial_grid, row_col='Riga', plot_col='Plot')

if {'MI_Garcia','MI_Visual'}.issubset(raw.columns):
    res_mi = regression_with_validation(raw, 'MI_Visual', 'MI_Garcia')
    moran_mi = export_validation_outputs(res_mi, 'mi')
    print('Cartographer MI ~ Manual MI')
    print(f"n = {len(res_mi['data'])}")
    print(f"Equation: y = {res_mi['slope']:.4f}x {res_mi['intercept']:+.4f}")
    print(f"Pearson r = {res_mi['r']:.4f}, p = {res_mi['p']:.3e}, R2 = {res_mi['r2']:.4f}, RMSE = {res_mi['rmse']:.4f}")
    print_validation_report(res_mi, moran_mi)
    plot_cv_comparison(res_mi, moran_mi, 'MI_Garcia', 'Cartographer MI', 'mi_cv_comparison.pdf')
    plot_observations_and_residuals(
        res_mi, moran_mi, 'MI_Garcia', 'Cartographer MI',
        'mi_observed_and_oof_residual_maps.pdf'
    )
    plot_regression(res_mi, 'MI_Visual', 'MI_Garcia', 'corr_fig2_MI_Garcia_vs_Visual_MI.pdf', 'Manual MI', 'Cartographer MI')
    # plot_regression(res_mi, 'MI_Visual', 'MI_Garcia', 'corr_fig2_MI_Garcia_vs_Visual_MI.pdf', 'Manual MI', 'Cartographer MI', identity=True)

    ba_mi = bland_altman(raw['MI_Garcia'], raw['MI_Visual'])
    print('Bland-Altman Cartographer MI - Manual MI')
    print(f"bias = {ba_mi['bias']:+.4f}, 95% LoA = [{ba_mi['loa_low']:.4f}, {ba_mi['loa_high']:.4f}]")
    print(f"Shapiro-Wilk W = {ba_mi['shapiro_w']:.4f}, p = {ba_mi['shapiro_p']:.3e}")
    plot_bland_altman(ba_mi, 'corr_fig5_bland_altman_MI_Garcia_vs_Visual.pdf', 'Mean of Cartographer MI and Manual MI', 'Cartographer MI - Manual MI')


Cartographer MI ~ Manual MI
n = 120
Equation: y = 2.0337x -2.4249
Pearson r = 0.9006, p = 1.626e-44, R2 = 0.8111, RMSE = 0.1782
Validation comparison using cumulative out-of-fold predictions:
        method_label  n_splits  oof_r2  oof_rmse  fold_r2_mean  fold_r2_sd  fold_rmse_mean  fold_rmse_sd
       Random 5-fold         5  0.8037    0.1817        0.7636      0.1371          0.1805        0.0233
Spatial block 5-fold         5  0.8017    0.1826        0.7560      0.1353          0.1790        0.0405
  Leave-one-Riga-out        20  0.8046    0.1813        0.6542      0.3480          0.1714        0.0606

Moran's I of cumulative OOF residuals (rook adjacency):
        method_label  moran_i  expected_i  p_permutation_two_sided
       Random 5-fold   0.2962     -0.0084                   0.0001
Spatial block 5-fold   0.3150     -0.0084                   0.0001
  Leave-one-Riga-out   0.3000     -0.0084                   0.0001


Bland-Altman Cartographer MI - Manual MI
bias = +0.0394, 95% LoA = [-0.4701, 0.5489]
Shapiro-Wilk W = 0.8928, p = 8.573e-08


## 8. Regression and agreement: Cartographer production measure vs field-measured production


In [9]:
prod = raw[['Riga', 'Plot', 'Production_plot_kg', 'N. frutti']].dropna().copy()
prod = prod.rename(columns={
    'Production_plot_kg': 'production_plot_kg',
    'Detected_Fruits': 'N. frutti'
})
print('Production descriptive statistics:')
print(prod.describe().round(2))
prod.describe().round(3).to_csv(TAB_DIR / 'production_descriptive_statistics.csv')

res_prod = regression_with_validation(prod, 'N. frutti', 'production_plot_kg')
moran_prod = export_validation_outputs(res_prod, 'production')
print('Field-measured production ~ Cartographer fruit detections')
print(f"n = {len(res_prod['data'])}")
print(f"Equation: predicted_production_kg = {res_prod['slope']:.4f} x detections {res_prod['intercept']:+.4f}")
print(f"Pearson r = {res_prod['r']:.4f}, p = {res_prod['p']:.3e}, R2 = {res_prod['r2']:.4f}, RMSE = {res_prod['rmse']:.4f} kg")
print_validation_report(res_prod, moran_prod)
plot_cv_comparison(
    res_prod, moran_prod, 'production_plot_kg', 'fruit weight / plot production (kg)',
    'production_cv_comparison.pdf'
)
plot_observations_and_residuals(
    res_prod, moran_prod, 'production_plot_kg', 'fruit weight / plot production (kg)',
    'production_observed_and_oof_residual_maps.pdf'
)
plot_regression(res_prod, 'N. frutti', 'production_plot_kg', 'corr_fig3_production_estimated_vs_measured.pdf', 'Detected Fruits', 'Harvested production (kg)', identity=False)
# plot_regression(res_prod, 'Detected_Fruits', 'production_plot_kg', 'corr_fig3_production_estimated_vs_measured_identity.pdf', 'Fruit per plant', 'Harvested production (kg)', identity=True, y_log=True, zoom_origin=True)

# detections = prod['Fruit_per_plant'].astype(float)
# measured = prod['measured_production_kg'].astype(float)
# predicted = res_prod['model'].predict(detections.to_frame())
# ba_prod = bland_altman(measured, predicted)
# print('Bland-Altman field-measured production - predicted production from calibration model')
# print(f"bias = {ba_prod['bias']:+.2f} kg, 95% LoA = [{ba_prod['loa_low']:.2f}, {ba_prod['loa_high']:.2f}] kg")
# print(f"Proportional bias: r = {ba_prod['prop_r']:.4f}, p = {ba_prod['prop_p']:.4f}")
# plot_bland_altman(ba_prod, 'corr_fig6_bland_altman_production.pdf', 'Mean Predicted Production from Calibration Model (kg)', 'Field-Measured Production (kg) - Predicted Production (kg)')


Production descriptive statistics:
         Riga    Plot  production_plot_kg  N. frutti
count  120.00  120.00              120.00     120.00
mean    10.50    3.50              207.18   75710.51
std      5.79    1.71               52.08   26184.95
min      1.00    1.00              122.40   30893.00
25%      5.75    2.00              174.03   54762.00
50%     10.50    3.50              196.03   72444.00
75%     15.25    5.00              223.58   86926.25
max     20.00    6.00              400.64  160409.00


Field-measured production ~ Cartographer fruit detections
n = 120
Equation: predicted_production_kg = 0.0019 x detections +64.1165
Pearson r = 0.9501, p = 1.512e-61, R2 = 0.9027, RMSE = 16.1746 kg
Validation comparison using cumulative out-of-fold predictions:
        method_label  n_splits  oof_r2  oof_rmse  fold_r2_mean  fold_r2_sd  fold_rmse_mean  fold_rmse_sd
       Random 5-fold         5  0.8995   16.4383        0.8920      0.0529         15.7174        5.3830
Spatial block 5-fold         5  0.8918   17.0555        0.8763      0.0570         16.6160        4.3009
  Leave-one-Riga-out        20  0.8949   16.8090        0.8115      0.2266         15.1596        7.4503

Moran's I of cumulative OOF residuals (rook adjacency):
        method_label  moran_i  expected_i  p_permutation_two_sided
       Random 5-fold   0.1079     -0.0084                   0.0786
Spatial block 5-fold   0.1950     -0.0084                   0.0039
  Leave-one-Riga-out   0.1310     -0.0084                   0

## 10. Manuscript-ready numerical summary

Copy these values into the LaTeX file after running the notebook.


In [10]:
print('=== MANUSCRIPT SUMMARY ===')
print(f'Analytical unit: {len(raw)} subsections after aggregating two canopy faces.')
print(f'Raw side-level observations: {len(raw)}.')
print(f'Total Cartographer-detected olives in aggregated dataset: {int(raw[CL_COLS].sum().sum())}.')
print('Ripening classes:')
for _, row in class_summary.iterrows():
    print(f"{row['class']}: {int(row['detected_fruit'])} ({row['percent']:.1f}%)")

if 'res_mi' in globals():
    print('MI regression:')
    print(f"Cartographer MI = {res_mi['slope']:.4f} * Manual MI {res_mi['intercept']:+.4f}; R2={res_mi['r2']:.4f}; RMSE={res_mi['rmse']:.4f}; spatial CV mean R2={res_mi['cv_r2_mean']:.4f}±{res_mi['cv_r2_sd']:.4f}; spatial CV mean RMSE={res_mi['cv_rmse_mean']:.4f}±{res_mi['cv_rmse_sd']:.4f}; pooled OOF R2={res_mi['cv_oof_r2']:.4f}; pooled OOF RMSE={res_mi['cv_oof_rmse']:.4f}")
    print(f"MI Bland-Altman bias={ba_mi['bias']:+.4f}; LoA=[{ba_mi['loa_low']:.4f}, {ba_mi['loa_high']:.4f}]; Shapiro p={ba_mi['shapiro_p']:.3e}")

if 'res_prod' in globals():
    print('Production regression:')
    print(f"Field-measured production = {res_prod['slope']:.4f} * Cartographer detections {res_prod['intercept']:+.4f}; R2={res_prod['r2']:.4f}; RMSE={res_prod['rmse']:.2f}; spatial CV mean R2={res_prod['cv_r2_mean']:.4f}±{res_prod['cv_r2_sd']:.4f}; spatial CV mean RMSE={res_prod['cv_rmse_mean']:.2f}±{res_prod['cv_rmse_sd']:.2f}; pooled OOF R2={res_prod['cv_oof_r2']:.4f}; pooled OOF RMSE={res_prod['cv_oof_rmse']:.2f}")

if 'ba_prod' in globals():
    print(f"Production Bland-Altman field-measured minus predicted production bias={ba_prod['bias']:+.2f} kg; LoA=[{ba_prod['loa_low']:.2f}, {ba_prod['loa_high']:.2f}] kg; proportional bias r={ba_prod['prop_r']:.4f}, p={ba_prod['prop_p']:.4f}")


=== MANUSCRIPT SUMMARY ===
Analytical unit: 120 subsections after aggregating two canopy faces.
Raw side-level observations: 120.
Total Cartographer-detected olives in aggregated dataset: 17950.
Ripening classes:
CL 0: 492982 (5.4%)
CL 1: 990521 (10.9%)
CL 2: 1949661 (21.5%)
CL 3: 5652095 (62.2%)
CL 4: 0 (0.0%)
MI regression:
Cartographer MI = 2.0337 * Manual MI -2.4249; R2=0.8111; RMSE=0.1782; spatial CV mean R2=0.7560±0.1353; spatial CV mean RMSE=0.1790±0.0405; pooled OOF R2=0.8017; pooled OOF RMSE=0.1826
MI Bland-Altman bias=+0.0394; LoA=[-0.4701, 0.5489]; Shapiro p=8.573e-08
Production regression:
Field-measured production = 0.0019 * Cartographer detections +64.1165; R2=0.9027; RMSE=16.17; spatial CV mean R2=0.8763±0.0570; spatial CV mean RMSE=16.62±4.30; pooled OOF R2=0.8918; pooled OOF RMSE=17.06
